# 05 Online MPC baseline

The MPC solves, at every 5-minute step, a convex problem over 144 steps with the same eq. (16) terms and weights as DPC
and the same terminal dead-zone term. It uses the same information as DPC (a persistence forecast: current load and PV
held over the horizon). Its first action passes through the same safety layer, and the plant is the same nonlinear
DistFlow simulation.

**Differences from the proposal, which apply to every MPC number on this site:**

1. The MPC uses a **convex LinDistFlow model**, not the non-convex AC OPF named in the proposal.
2. Its voltage limits are **soft** (squared slack, weight 1000), as in the DPC loss.
3. Its SoC model is **lossless** (one signed P per battery); the plant keeps a charge and discharge efficiency of 0.95.
4. It was **not tuned**: same weights as DPC, nothing chosen on validation.

Any speed comparison of DPC on this site is therefore against a convex MPC, not against an AC-OPF MPC.

```{include} ../_caveats.md
```

## Why the SoC model is lossless (superseded pilots)

The first MPC version modelled efficiency by splitting P into charge and discharge parts. On the two pilot days it charged
and discharged at the same time, which in its own model burns stored energy. A pre-specified diagnostic and a fixed small
penalty did not remove this, so the model was changed to one signed P per battery, before any comparison with DPC existed.
The numbers below belong to the **superseded** versions and are shown only to document that change.

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
p1, ov, pp = load("m4_mpc_pilot.json"), load("m4_mpc_overlap_diagnostic_rerun.json"), load("m4_mpc_pilot_penalty.json")
for rec, name in [(p1, "m4_mpc_pilot.json"), (ov, "m4_mpc_overlap_diagnostic_rerun.json"), (pp, "m4_mpc_pilot_penalty.json")]:
    display(Markdown(provenance(rec, name)))
rows = []
for reg, label in REGIMES.items():
    o = ov["regimes"][reg]
    rows.append({"regime": label,
                 "simultaneous charge+discharge, first pilot": pct(p1["regimes"][reg]["share_battery_steps_simultaneous_charge_discharge"], 1),
                 "mean overlap (MW), diagnostic": fmt(o["overlap_mw"]["mean"], 3),
                 "mean overlap (MW), with penalty": fmt(pp["regimes"][reg]["overlap_mw"]["mean"], 3)})
table(rows, index="regime")

## Lossless pilot: pre-specified pass criteria

In [ ]:
lp = load("m4_mpc_pilot_lossless.json")
display(Markdown(provenance(lp, "m4_mpc_pilot_lossless.json")))
c = lp["criteria"]
rows = []
for reg, label in REGIMES.items():
    r = lp["regimes"][reg]
    rows.append({"regime": label, "steps": r["steps"], f"optimal (≥ {pct(c['min_optimal_share'], 1)})": pct(r["optimal_share"], 1),
                 "no charge/discharge split": fmt(r["no_simultaneous_by_construction"]),
                 f"max one-step SoC mismatch (fail above {fmt(c['mismatch_bound'] + c['mismatch_slack'], 4)})": fmt(r["soc_mismatch_1_step"]["max"], 3),
                 "safety layer changed the action": pct(r["share_battery_steps_safety_layer_changes_action"])})
display(table(rows, index="regime"))
display(Markdown(f"All criteria pass: **{fmt(lp['all_criteria_pass'])}**."))

## Full validation run: persistence MPC next to DPC and the droops

All controllers below run on exactly the same validation steps, in the continuous loop with 15 solver sweeps.
DPC: median and range over seeds 0, 1 and 2.

In [ ]:
mpc = {reg: load(f"m4_mpc_full/{reg}.json") for reg in REGIMES}
display(Markdown(provenance(load("m4_mpc_pilot_lossless.json"), "m4_mpc_full/ (same commit as the lossless pilot)")))
sd = load("m3_exp2_seeds.json"); d2 = load("m3_droop_tuning_round2.json")

def summary_row(name, reg, r):
    s = r["summary"]
    return {"controller": name, "regime": REGIMES[reg], "objective": fmt(r["objective"]),
            "VVR": pct(s["vvr_share_of_steps_with_any_violation"], 3), "V-RMSE (pu)": fmt(s["v_rmse_pu"]),
            "losses (kW)": fmt(s["mean_network_loss_kw"], 1), "mean |P| (MW)": fmt(s["mean_abs_p_mw"], 4),
            "mean |Q| (Mvar)": fmt(s["mean_abs_q_mvar"], 3)}

rows = []
for reg in REGIMES:
    rows.append(summary_row("no battery action", reg, sd["references"]["no_battery_action"]["regimes"][reg]))
    rows.append(summary_row("droop, original", reg, sd["references"]["volt_var_droop"]["regimes"][reg]))
    t = d2["validation"]["droop_final_tuned"]["regimes"][reg]
    rows.append({"controller": "droop, tuned (final)", "regime": REGIMES[reg], "objective": fmt(t["objective"]),
                 "VVR": pct(t["vvr_share_of_steps_with_any_violation"], 3), "losses (kW)": fmt(t["mean_network_loss_kw"], 1),
                 "mean |Q| (Mvar)": fmt(float(np.mean(t["mean_abs_q_mvar"])), 3)})
    rows.append(summary_row("persistence MPC", reg, mpc[reg]))
    seeds = [sd["candidates"][s]["regimes"][reg] for s in sd["candidates"]]
    def mr(get, digits=5, as_pct=False):
        v = np.array([get(x) for x in seeds])
        f = (lambda y: pct(y, 3)) if as_pct else (lambda y: fmt(float(y), digits))
        return f"{f(np.median(v))} [{f(v.min())}-{f(v.max())}]"
    rows.append({"controller": "DPC, seeds 0/1/2", "regime": REGIMES[reg], "objective": mr(lambda x: x["objective"]),
                 "VVR": mr(lambda x: x["summary"]["vvr_share_of_steps_with_any_violation"], as_pct=True),
                 "V-RMSE (pu)": mr(lambda x: x["summary"]["v_rmse_pu"]),
                 "losses (kW)": mr(lambda x: x["summary"]["mean_network_loss_kw"], 1),
                 "mean |P| (MW)": mr(lambda x: x["summary"]["mean_abs_p_mw"], 4),
                 "mean |Q| (Mvar)": mr(lambda x: x["summary"]["mean_abs_q_mvar"], 3)})
table(rows).set_index(["regime", "controller"]).fillna("—")

*— means the value is not stored in that record.*

In [ ]:
rel = {}
for reg in REGIMES:
    dpc = float(np.median([sd["candidates"][s]["regimes"][reg]["objective"] for s in sd["candidates"]]))
    rel[reg] = (dpc, mpc[reg]["objective"], 100 * (dpc - mpc[reg]["objective"]) / mpc[reg]["objective"])
display(Markdown("On the validation objective, DPC (median over seeds) is " +
                 " and ".join(f"{abs(p):.1f}% {'lower' if p < 0 else 'higher'} than the persistence MPC in the {REGIMES[reg]} regime "
                              f"({d:.5f} against {m:.5f})" for reg, (d, m, p) in rel.items()) +
                 ". This is a validation result: the DPC variant and its seeds were chosen on the validation part, while the MPC "
                 "was not tuned."))
lp_h = load("m4_mpc_pilot_lossless.json")["horizon"] * load("m2_summary.json")["config"]["time"]["dt_min"] / 60
display(Markdown(f"The persistence forecast holds the current load and PV constant for {lp_h:g} hours, which is a crude forecast; a "
                 "better forecast would probably improve the MPC. The oracle MPC on page 06 shows the upper bound."))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, reg in zip(axes, REGIMES):
    items = [("droop, tuned", d2["validation"]["droop_final_tuned"]["regimes"][reg]["objective"]),
             ("persistence MPC", mpc[reg]["objective"])]
    items += [(f"DPC seed {i}", sd["candidates"][f"seed_{i}"]["regimes"][reg]["objective"]) for i in range(3)]
    ax.barh([n for n, _ in items], [v for _, v in items], color=[SERIES[3], SERIES[1], SERIES[0], SERIES[0], SERIES[0]], height=0.6)
    for y, (_, v) in enumerate(items):
        ax.text(v, y, f" {v:.5f}", va="center", fontsize=8, color=INK)
    ax.invert_yaxis(); ax.set_xlabel("validation objective (lower is better)"); ax.set_title(REGIMES[reg], loc="left")
    ax.set_xlim(0, max(v for _, v in items) * 1.25)
fig.tight_layout(); plt.show()
display(Markdown("No battery action and the original droop are left out of the chart because their objectives are much "
                 "larger; they are in the table above."))

## MPC solver statistics (full run)

In [ ]:
rows = []
for reg, label in REGIMES.items():
    m = mpc[reg]; s = m["solve_seconds"]
    rows.append({"regime": label, "steps": m["steps"], "solver statuses": ", ".join(f"{k}: {v}" for k, v in m["solver_statuses"].items()),
                 "solve time median / p95 / max (s)": f"{s['median']:.3f} / {s['p95']:.3f} / {s['max']:.2f}",
                 "safety layer changed the action": pct(m["share_battery_steps_safety_layer_changes_action"]),
                 "end-of-day SoC": f"{fmt(m['end_of_day_soc_min'], 2)} - {fmt(m['end_of_day_soc_max'], 2)}"})
display(table(rows, index="regime"))
display(Markdown("A latency study of DPC against MPC at different horizons and network sizes has not been done yet."))